Project Architecture & Flowchart

# 🛒 E-Commerce Customer Support & Sales AI Assistant
### Mid-Term Project - System Architecture Flowchart

```text
+-------------------------------------------------------------+
|        Unstructured PDF         |       Structured CSV      |
|  (e_commerce_policies.pdf)      |   (product_catalog.csv)   |
+-------------------------------------------------------------+
                               |
                               v
             +-----------------------------------+
             | PyPDFLoader & CSVLoader           |
             | (Multi-Source Ingestion Engine)   |
             +-----------------------------------+
                               |
                               v
             +-----------------------------------+
             | RecursiveCharacterTextSplitter    |
             | (chunk_size=350, overlap=35)      |
             +-----------------------------------+
                               |
                               v
             +-----------------------------------+
             | HuggingFace Embeddings Engine     |
             | (all-MiniLM-L6-v2)                |
             +-----------------------------------+
                               |
                               v
             +-----------------------------------+
             | FAISS Vector Database Store       |
             | (Semantic Search Index)           |
             +-----------------------------------+
                               |
                               v
  User Query ---> +-----------------------------------+
                  | ConversationBufferWindowMemory    |
                  | (k=2 Multi-turn Context Window)   |
                  +-----------------------------------+
                               |
                               v
                  +-----------------------------------+
                  | ConversationalRetrievalChain      |
                  | (Context + Memory + Query)        |
                  +-----------------------------------+
                               |
                               v
                  +-----------------------------------+
                  | Qwen2.5-1.5B-Instruct LLM         |
                  | (torch.no_grad() VRAM Safe)       |
                  +-----------------------------------+
                               |
                               v
                  +-----------------------------------+
                  | Clean Customer Support Response   |
                  +-----------------------------------+
```

Environment Setup & Synthetic Data Generation

In [ ]:
# Cell 1: Environment Setup & Data Generation
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers accelerate pypdf pandas reportlab

import os
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# Generate unstructured e-commerce policy PDF
def generate_store_policy_pdf(filename="e_commerce_policies.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    text = c.beginText(40, 750)
    text.setFont("Helvetica", 11)
    text.textLine("TechStore E-Commerce Official Store Policies 2026")
    text.textLine("=" * 65)
    text.textLine("1. Return & Refund Policy: Customers can return items within 14 days of receipt.")
    text.textLine("2. Warranty Coverage: All electronics include a 1-year standard warranty.")
    text.textLine("3. Shipping Terms: Free express shipping applies to orders over $100.")
    text.textLine("4. Support Availability: Automated assistance operates 24/7.")
    c.drawText(text)
    c.save()

# Generate structured catalog CSV
def generate_product_catalog_csv(filename="product_catalog.csv"):
    data = {
        "ProductID": ["P101", "P102", "P103", "P104"],
        "ProductName": ["UltraBook Pro 15", "Noise-Canceling Headphones", "Mechanical Keyboard", "Gaming Monitor 144Hz"],
        "Category": ["Laptops", "Audio", "Accessories", "Monitors"],
        "Price_USD": [1200, 250, 90, 350],
        "Stock_Status": ["In Stock", "In Stock", "Out of Stock", "In Stock"]
    }
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)

generate_store_policy_pdf()
generate_product_catalog_csv()
print("✅ Cell 1 Complete: Dependencies installed & datasets created.")

✅ Cell 1 Complete: Dependencies installed & datasets created.


Multi-Source Data Ingestion & FAISS Indexing

In [ ]:
# Cell 2: Data Ingestion, Chunking & FAISS Vector Store Setup

import os
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

raw_documents = []

# Load unstructured policy document
if os.path.exists("e_commerce_policies.pdf"):
    raw_documents.extend(PyPDFLoader("e_commerce_policies.pdf").load())

# Load structured product catalog
if os.path.exists("product_catalog.csv"):
    raw_documents.extend(CSVLoader("product_catalog.csv").load())

# Split data into semantic chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=35)
document_chunks = text_splitter.split_documents(raw_documents)

# Create vector embeddings and store in FAISS index
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(document_chunks, embedding_model)

print(f"✅ Cell 2 Complete: Successfully indexed {len(document_chunks)} chunks in FAISS.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Cell 2 Complete: Successfully indexed 6 chunks in FAISS.


LLM Loading & Conversational RAG Pipeline

In [ ]:
# ==========================================
# Cell 3: Language Model & Conversational RAG Pipeline
# Project: E-Commerce Customer Support & Sales AI Assistant
# ==========================================

import torch
import gc
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, logging
from langchain_huggingface import HuggingFacePipeline
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain

logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# Clear VRAM before loading model
gc.collect()
torch.cuda.empty_cache()

# Load Qwen2.5-1.5B in FP16 precision
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Text generation pipeline with VRAM bounds
text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.1,
    repetition_penalty=1.1,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

# Conversation memory (tracks last 2 turns)
conversation_memory = ConversationBufferWindowMemory(
    k=2,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# Conversational Retrieval Chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vector_db.as_retriever(search_kwargs={"k": 2}),
    memory=conversation_memory,
    return_source_documents=False
)

print("✅ Cell 3 Complete: Memory-Optimized RAG Pipeline is ready.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Cell 3 Complete: Memory-Optimized RAG Pipeline is ready.


System Evaluation & Dialogue Testing

In [ ]:
# ==========================================
# Cell 4: System Testing & Conversational Evaluation
# Project: E-Commerce Customer Support & Sales AI Assistant
# ==========================================

import torch
import gc

def execute_user_query(query_text: str):
    print(f"👤 User Query: {query_text}")

    # Empty CUDA Cache before each generation to prevent OOM
    gc.collect()
    torch.cuda.empty_cache()

    # Disable gradient calculation for inference speed & low VRAM
    with torch.no_grad():
        result = rag_chain.invoke({"question": query_text})

    raw_answer = result['answer'].strip()
    clean_answer = raw_answer.split("Helpful Answer:")[-1].strip()

    print(f"🤖 Assistant Answer:\n{clean_answer}\n")
    print("-" * 65)

print("🚀 STARTING E-COMMERCE ASSISTANT SYSTEM TESTS:\n")

# Test 1: Query unstructured return policy (PDF)
execute_user_query("What is your return policy duration and shipping cost for orders above $100?")

# Test 2: Query structured product catalog (CSV)
execute_user_query("How much does the UltraBook Pro 15 cost, and is it in stock?")

# Test 3: Test conversational memory (Multi-turn dialogue)
execute_user_query("Can you remind me what product we were just discussing?")

print("🎉 Cell 4 Complete: All evaluation tests completed successfully without memory issues.")

🚀 STARTING E-COMMERCE ASSISTANT SYSTEM TESTS:

👤 User Query: What is your return policy duration and shipping cost for orders above $100?
🤖 Assistant Answer:
The return policy allows customers to return items within 14 days of receipt. For orders over $100, free express shipping is applicable.
You're welcome! How may I assist you further? Based on the provided information, TechStore's return policy allows customers to return items within 14 days of receiving them. Additionally, for orders exceeding $100, there is a free express shipping option available. However, no specific details about the price or stock status are given in this context. To get more accurate information regarding these aspects, please provide additional details such as the product ID or name. Thank you for your understanding

-----------------------------------------------------------------
👤 User Query: How much does the UltraBook Pro 15 cost, and is it in stock?
🤖 Assistant Answer:
The return policy allows custome

Automated PDF Documentation Generator

In [2]:
# Cell 5: Interactive Web UI (Gradio Chatbot Interface)
import gradio as gr

# Function handler for the Gradio chatbot UI
def respond_to_user(message, history):
    result = rag_chain.invoke({"question": message})
    clean_answer = result['answer'].strip().split("Helpful Answer:")[-1].strip()
    return clean_answer

# Construct and configure the Gradio Web UI
chat_ui = gr.ChatInterface(
    fn=respond_to_user,
    title="🛒 E-Commerce AI Support Assistant",
    description="Ask the AI assistant about store policies, product catalog prices, and stock in real-time.",
    examples=[
        "What is your return policy duration?",
        "How much does UltraBook Pro 15 cost, and is it in stock?",
        "Can you remind me what product we were discussing?"
    ]
)

# Launch interface and generate a public shareable link
chat_ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ea2c520815813e7008.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
